# YOLO Object Detection — Data

This notebook loads the COCO dataset and builds a custom PyTorch `Dataset` used by the training loop.

## Dependencies

Install and import the packages we need, then select the device (GPU via MPS on Apple Silicon, or CPU).

In [48]:
# --- Installing Dependencies ---

# Installing PyTorch
%pip install --upgrade pip -q
%pip install torch torchvision -q
%pip install matplotlib

import torch
import torchvision
import json
import pprint
import matplotlib.pyplot as plt
import matplotlib.patches
from PIL import Image
import numpy as np
import os

# --- Setup Dependencies ---
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") # tells pytorch to use mps (Metal Performance Shaders) which is the GPU if available, if not use the CPUs
print(f"PyTorch is using {device}.")

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
PyTorch is using mps.


## Getting the Data

COCO doesn't support automatic download via PyTorch, so the images and annotations must be downloaded manually from [cocodataset.org](https://cocodataset.org/#download).

We need two things:
- **`val2017/`** — folder of images
- **`annotations/instances_val2017.json`** — bounding box labels for every image

In [21]:
images_path = "val2017"
labels_path = "annotations/instances_val2017.json"

with open(labels_path) as f:
    coco = json.load(f)

# We can see the information stored in the labels here:                                                                                                               
pp = pprint.PrettyPrinter(indent=2)
                                                                                                                                                    
print("=== Keys ===")                                                                                                                               
print(list(coco.keys()))                                                                                                                            
                                                                                                                                                    
print("\n=== Info ===")
pp.pprint(coco["info"])

print("\n=== Sample Image ===")
pp.pprint(coco["images"][0])
                                                                                                                                                    
print("\n=== Sample Annotation ===")
pp.pprint(coco["annotations"][0])                                                                                                                   
                
print("\n=== Categories ===")
pp.pprint(coco["categories"])

=== Keys ===
['info', 'licenses', 'images', 'annotations', 'categories']

=== Info ===
{ 'contributor': 'COCO Consortium',
  'date_created': '2017/09/01',
  'description': 'COCO 2017 Dataset',
  'url': 'http://cocodataset.org',
  'version': '1.0',
  'year': 2017}

=== Sample Image ===
{ 'coco_url': 'http://images.cocodataset.org/val2017/000000397133.jpg',
  'date_captured': '2013-11-14 17:02:52',
  'file_name': '000000397133.jpg',
  'flickr_url': 'http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg',
  'height': 427,
  'id': 397133,
  'license': 4,
  'width': 640}

=== Sample Annotation ===
{ 'area': 702.1057499999998,
  'bbox': [473.07, 395.93, 38.65, 28.67],
  'category_id': 18,
  'id': 1768,
  'image_id': 289343,
  'iscrowd': 0,
  'segmentation': [ [ 510.66,
                      423.01,
                      511.72,
                      420.03,
                      510.45,
                      416.0,
                      510.34,
                      413.02,
        

### Understanding the COCO Format

The annotation JSON is a dictionary with these keys:

| Key | Contents |
|---|---|
| `info` | Dataset metadata (version, year, contributor) |
| `images` | List of images — each has `id`, `file_name`, `width`, `height` |
| `annotations` | List of labelled objects — each has `bbox`, `category_id`, `image_id`, `iscrowd` |
| `categories` | The 80 class names, each with an `id` and `name` |

**Key fields in each annotation:**
- `bbox` — `[x, y, width, height]` in pixels (top-left corner)
- `category_id` — class ID (1–90, not contiguous — we remap these to 0–79)
- `iscrowd` — if `1`, skip it (crowd annotations are imprecise)
- `segmentation` — polygon outline, not needed for detection

## Building the Dataset

`torch.utils.data.Dataset` is PyTorch's interface for datasets. We subclass it and implement two methods:
- `__len__` — returns the number of images
- `__getitem__` — returns one `(image, labels)` pair given an index

**Label format we produce:** each object becomes `[class_idx, cx, cy, w, h]` where all values are normalised to `[0, 1]` relative to image size. COCO category IDs jump from 1–90 with gaps, so we remap them to clean 0–79 indices using `cat_to_idx`.

In [ ]:
class coco_dataset(torch.utils.data.Dataset):

    def __init__(self, images_path = "val2017" ,image_size = 416, labels_path = "annotations/instances_val2017.json"):
        super().__init__()
        self.images_path = images_path
        self.labels_path = labels_path
        self.image_size = image_size
    
        with open(labels_path) as f:
            coco = json.load(f)

        self.cat_to_idx = {cat['id']: idx for idx, cat in enumerate(coco['categories'])}
        
        self.labels = {}
        for label in coco["annotations"]:
            self.labels.setdefault(label["image_id"], []).append(label)
        
        self.images = coco["images"]
        
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, index):
        img_info = self.images[index]
        img_w, img_h = img_info['width'], img_info['height']
                                                                                                                                                                                                        
        img_path = os.path.join(self.images_path, img_info['file_name'])
        image = Image.open(img_path).convert('RGB')                                                                                                                                                      
        image = image.resize((self.image_size, self.image_size))                                                                                                                                         
        image = torch.from_numpy(np.array(image)).permute(2, 0, 1).float() / 255.0
                                                                                                                                                                                                        
        labels = [] 
        for label in self.labels.get(img_info['id'], []):                                                                                                                                                
            if label['iscrowd']:
                continue
            x, y, w, h = label['bbox']                                                                                                                                                                   
            cx = (x + w / 2) / img_w
            cy = (y + h / 2) / img_h
            class_idx = self.cat_to_idx[label['category_id']]
            labels.append([class_idx, cx, cy, w / img_w, h / img_h])
                                                                                                                                                                                                        
        labels = torch.tensor(labels, dtype=torch.float32) if labels else torch.zeros((0, 5))                                                                                                            
                                                                                                                                                                                                        
        return image, labels

dataset = coco_dataset(images_path=images_path, labels_path=labels_path) 
print(f"Dataset size: {len(dataset)}")

image, labels = dataset[0]
print(f"\nImage tensor shape: {image.shape}")
print(f"Labels tensor shape: {labels.shape}")
print(f"Sample labels:\n{labels}")

plt.imshow(image.permute(1,2,0))
plt.axis('off')
plt.show()

H, W = image.shape[1], image.shape[2]                                                                                                                                                                
   
# Convert image from float 0-1 to uint8 0-255                                                                                                                                                        
image_uint8 = (image * 255).byte()

# Convert labels from [class, cx, cy, w, h] normalized → [xmin, ymin, xmax, ymax] pixels                                                                                                             
boxes = []
for label in labels:                                                                                                                                                                                 
    _, cx, cy, w, h = label
    xmin = (cx - w / 2) * W
    ymin = (cy - h / 2) * H                                                                                                                                                                          
    xmax = (cx + w / 2) * W
    ymax = (cy + h / 2) * H                                                                                                                                                                          
    boxes.append([xmin, ymin, xmax, ymax])                                                                                                                                                           

boxes = torch.tensor(boxes, dtype=torch.float32)                                                                                                                                                     
                
result = torchvision.utils.draw_bounding_boxes(image_uint8, boxes)                                                                                                                                   

plt.imshow(result.permute(1, 2, 0))                                                                                                                                                                  
plt.axis('off') 
plt.show()